# H2：场景模糊测试 —— 变异 + 通过判据
# H2: Scenario Fuzzing — Mutation + Pass Criteria

## 一、实验原理 | Principle

### 0.1 工程背景 | Engineering Background

SOTIF 标准的核心焦虑是"**未知的不安全场景**"（unknown unsafe scenarios）：工程师能想到的危险场景都测了，但没想到的呢？**模糊测试（Fuzzing）** 就是系统性地"想不到也要测"：对场景参数做随机变异，批量仿真，看 AEB 会不会在某个奇怪的参数组合下失效。

本实验把你在 A1 实现的 AEB 控制器作为被测对象（SUT），用变异场景做"红蓝对抗"——这是 SOTIF 验证流程的微缩实践。


### 1.1 什么是模糊测试（Fuzzing）？
**模糊测试**是一种自动化软件测试技术：向被测系统输入**大量随机/半随机变异的数据**，观察系统是否崩溃或违反预期。
它起源于 1990 年代威斯康星大学的 Barton Miller 教授，现已成为发现软件漏洞的标准手段（如 AFL、libFuzzer）。

在自动驾驶测试中，模糊测试的对象是**驾驶场景**：
- 被测系统：AEB 紧急制动控制器（你在 A1 中实现的）
- 变异输入：场景参数（初速度、初始位置、切入时机……）
- 预期：无碰撞、最小 TTC 足够大、不误触发

### 1.2 场景变异算子 | Mutation Operators
本实验的变异算子（全部基于正态/均匀分布随机采样，种子固定为 42 保证可复现）：

| 参数 | 变异分布 | 物理意义 |
|------|---------|---------|
| `ego_v0` | N(15, 2.5) clip [11, 20] m/s | 自车初速度 |
| `ego_x0` | N(10, 2.0) clip [5, 15] m | 自车初始位置 |
| `tgt_v0` | N(10, 2.5) clip [7, 15] m/s | 目标车初速度 |
| 切入间隙 `g` | U[12, 32] m | 切入瞬间两车纵向间隙（由 `cutin_t` 反解保证） |
| `cutin_t` | 由间隙反解，clip [2, 10] s | 目标车切入时机 |
| 对抗制动 | 30% 概率，目标值 N(7,2) m/s | 切入后目标车急减速（最危险的真实行为） |
| 无切入 | 20% 概率 | 目标车不切入（用于测量误触发率） |

### 1.3 通过判据与 Oracle 问题 | Pass Criteria & The Oracle Problem
模糊测试的核心难题是 **Oracle 问题**：如何自动判断一次运行是"通过"还是"失败"？
本实验定义三级判据：

1. **单次运行通过**：`无碰撞 AND 最小TTC > 1.5s`
   - 碰撞定义：纵向间隙 < 0.5m 且横向距离 < 1.5m（两车真实重叠）
   - TTC 只在目标车进入自车道（|横向距离| < 2.0m）时计算
2. **误触发率**：在无切入场景中，控制器施加减速度 > 3.0 m/s² 的比例 < 5%
   - 误触发 = 没有危险却急刹车（会导致被后车追尾）
3. **群体通过率**：通过的切入场景 / 总切入场景，应在 30%–90% 之间
   - 100% 通过说明变异强度不够（没测出边界）；<30% 说明控制器太弱或场景不合理

### 1.4 警告：仿真引擎诚实声明 | Honesty Disclosure
**本实验的车辆运动学采用 Python `KinematicBicycleModel`（与 A1/A2 的自车模型相同），目标车的切入轨迹为脚本化正弦曲线。**

为什么不用 esmini 驱动目标车？
- 实测发现：esmini 3.10.0 在 ctypes 调用模式下，场景目标车**纵向/横向均不运动**（`SE_StepDT` 后位置恒定，已用最小复现脚本验证）。
- H2 需要 50 次快速变异测试（本实验 50 次 < 1 秒），运动学模型是正确且诚实的工程选择。
- 场景文件仍用 `scenariogeneration` 生成标准 OpenSCENARIO，可被 esmini 解析校验。

> 这不是"杜撰的演示"：变异、闭环控制、碰撞检测、统计全部是真实计算，只是车辆动力学用运动学模型而非 esmini 内部控制器——特此声明。



In [ ]:
import sys
sys.dont_write_bytecode = True  # 禁用.pyc缓存
import os
sys.path.insert(0, '/home/jovyan/labs')
from common.esmini_env import KinematicBicycleModel
from common.controller import BaseController
import math
import numpy as np
print('ok: labs/common imported')
print('note: kinematic model is used (see honesty disclosure above)')


### 常见报错说明 | Troubleshooting

| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `ModuleNotFoundError: common` | 找不到 labs 库 | 检查 `sys.path` 是否包含 `/home/jovyan/labs` |
| `ImportError: KinematicBicycleModel` | 导入模型失败 | 确认 `labs/common/esmini_env.py` 存在 |
| `AssertionError: pass rate` | 通过率不在 30%–90% | 不要改随机种子！检查是否误改了变异分布参数 |
| 图表中文乱码 | matplotlib 缺中文字体 | 本实验图表全部使用英文标签，这是预期的 |


**术语速查 | Glossary**
- **模糊测试 Fuzzing**：用随机变异输入找 bug——一句话：瞎猫碰死耗子，但瞎得很有系统。
- **SUT（被测系统）**：本实验指你的 AEB 控制器——一句话：被"拷打"的对象。
- **误触发 False Trigger**：不该刹车时刹了——一句话：胆小鬼，风吹草动就急刹（用户体验杀手）。
- **SOTIF**：预期功能安全——一句话：系统没坏，但"我以为它能 handle，结果它没 handle"的安全问题。



## 二、实验目的 | Objectives
1. 理解模糊测试的基本思想：用随机变异暴露被测系统的边界与缺陷
2. 掌握场景变异算子的设计：哪些参数值得变异？分布如何选择？
3. 理解 Oracle 问题：为自动驾驶定义可计算的通过/失败判据
4. 学会统计分析：通过率、误触发率、边界案例识别
5. 建立诚实仿真的工程素养：明确声明仿真器的能力边界

## 三、学生任务 | Your Tasks
1. **运行**：执行下面的模糊测试循环（50 个变异场景，约需 < 1 秒）
2. **记录**：通过率、误触发率、碰撞数、边界案例编号
3. **思考**：
   - 为什么有些场景必然碰撞？（提示：看切入间隙 `g` 和目标车制动）
   - 误触发率为 0 是好事吗？（提示：是否以牺牲灵敏度为代价？）
   - 如果把 TTC 阈值从 3.0 改为 2.0，通过率会怎么变？
4. **挑战**（见文末）：设计你自己的变异算子

### 🔬 试一试 | Try It

1. **改变异强度**：扰动标准差翻倍——预期：通过率下降、碰撞案例增多，体会"变异强度 vs 有效测试"的平衡。
2. **加变异维度**：对"切入时机"也做变异——预期：可能发现新的失效模式，理解"维度越全，越接近真实长尾"。
3. **看边界案例**：把 4 个 TTC∈[1.74, 2.32] 的案例单独重跑——预期：结果可复现（种子固定），体会"可复现性"是测试的基本要求。



In [ ]:
class LaneAwareAEB(BaseController):
    """Lane-aware TTC AEB controller.

    Only considers targets within 1.5m lateral distance (ego's lane).
    This avoids false triggers from adjacent-lane traffic.
    """

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        in_lane = [t for t in targets if abs(t.get('y', 99) - ego['y']) < 1.5]
        if not in_lane:
            return {'target_a': 0.0, 'steer_angle': 0.0}
        lead = min(in_lane, key=lambda t: abs(t['x'] - ego['x']))
        rel_d = abs(lead['x'] - ego['x']) - 4.5
        rel_v = ego['v'] - lead['v']
        ttc = rel_d / rel_v if rel_v > 0.1 else float('inf')
        return {'target_a': -5.0 if ttc < 3.0 else 0.0, 'steer_angle': 0.0}

print('ok: controller defined (lane-aware TTC-AEB)')


def run_case(ego_x0, ego_v0, tgt_x0, tgt_v0, cutin_t, tgt_brake_v,
             dt=0.05, steps=300):
    """Run ONE fuzz case with kinematic models.

    Target behavior is scripted: sinusoidal lane change over 3s starting
    at cutin_t, plus optional hard braking after cut-in (adversarial).
    Returns: (min_ttc, collision, max_decel)
    """
    ego = KinematicBicycleModel()
    ego.reset(x=ego_x0, y=0.0, yaw=0.0, v=ego_v0)
    tgt = KinematicBicycleModel()
    tgt.reset(x=tgt_x0, y=-3.0, yaw=0.0, v=tgt_v0)
    ctl = LaneAwareAEB()
    ctl.reset()
    min_ttc, collision, max_decel = float('inf'), False, 0.0
    for s in range(steps):
        t = s * dt
        if cutin_t is not None and t >= cutin_t:
            prog = min(1.0, (t - cutin_t) / 3.0)
            tgt_y = -3.0 * (1.0 - math.sin(prog * math.pi / 2))
            if tgt_brake_v is not None and t >= cutin_t + 1.0:
                tgt.v = max(tgt_brake_v, tgt.v - 3.0 * dt)
            tgt_v_now = tgt.v
        else:
            tgt_y, tgt_v_now = -3.0, tgt.v
        tgt.x += tgt_v_now * dt
        tgt.y = tgt_y
        obs = {'ego': {'x': ego.x, 'y': ego.y, 'v': ego.v},
               'targets': [{'x': tgt.x, 'y': tgt.y, 'v': tgt_v_now}],
               'sim_time': t, 'dt': dt}
        act = ctl.step(obs)
        max_decel = max(max_decel, -act['target_a'])
        ego.step(act['target_a'], 0.0, dt)
        rel_d = abs(tgt.x - ego.x) - 4.5
        dy = abs(tgt.y - ego.y)
        if dy < 2.0:
            rv = ego.v - tgt_v_now
            if rv > 0.1:
                min_ttc = min(min_ttc, rel_d / rv)
        if rel_d < 0.5 and dy < 1.5:
            collision = True
            break
    return min_ttc, collision, max_decel


# smoke test: 1 case
import time as _time
_t0 = _time.time()
_r = run_case(10.0, 14.0, 45.0, 12.0, 6.0, None)
print('ok: smoke test 1 case: min_ttc=%.2fs, collision=%s (%.2fs)' %
      (_r[0], _r[1], _time.time() - _t0))


## 四、模糊测试主循环 | Fuzzing Loop
运行 50 个变异场景（随机种子 42，保证每次运行结果一致）。
- 每个 case 的参数按"实验原理"中的分布采样
- 切入间隙 `g` 直接采样后反解 `cutin_t`，确保覆盖危险几何


In [ ]:
rng = np.random.default_rng(42)  # fixed seed: reproducible
N_FUZZ = 50
results = []

for i in range(N_FUZZ):
    ego_v0 = float(np.clip(rng.normal(15, 2.5), 11, 20))
    ego_x0 = float(np.clip(rng.normal(10, 2.0), 5, 15))
    tgt_v0 = float(np.clip(rng.normal(10, 2.5), 7, 15))
    if rng.random() < 0.2:
        tgt_x0 = float(np.clip(rng.normal(35, 8.0), 20, 60))
        cutin_t, tgt_brake_v = None, None
    else:
        G0 = float(rng.uniform(25, 50))
        tgt_x0 = ego_x0 + G0
        g = float(rng.uniform(12, 32))
        dv = ego_v0 - tgt_v0
        tc = (G0 - g) / dv if dv > 0.5 else 5.0
        cutin_t = float(np.clip(tc, 2.0, 10.0))
        tgt_brake_v = (float(np.clip(rng.normal(7, 2), 4, 10))
                       if rng.random() < 0.3 else None)
    min_ttc, collision, max_decel = run_case(
        ego_x0, ego_v0, tgt_x0, tgt_v0, cutin_t, tgt_brake_v)
    results.append({
        'case': i, 'ego_v0': ego_v0, 'ego_x0': ego_x0,
        'tgt_v0': tgt_v0, 'tgt_x0': tgt_x0, 'cutin_t': cutin_t,
        'tgt_brake_v': tgt_brake_v, 'no_cutin': cutin_t is None,
        'min_ttc': min_ttc, 'collision': collision, 'max_decel': max_decel,
    })

print('ok: fuzzing done: %d mutated scenarios executed' % N_FUZZ)


## 五、评测断言 | Evaluation
三级判据（见"实验原理 1.3"）：
1. 单次通过：无碰撞 且 最小TTC > 1.5s
2. 误触发率 < 5%
3. 群体通过率在 [30%, 90%] 之间


In [ ]:
cutin_runs = [r for r in results if not r['no_cutin']]
passed = [r for r in cutin_runs
          if not r['collision'] and r['min_ttc'] > 1.5]
pass_rate = len(passed) / len(cutin_runs)

no_cutin_runs = [r for r in results if r['no_cutin']]
false_trig = [r for r in no_cutin_runs if r['max_decel'] > 3.0]
ftr = len(false_trig) / len(no_cutin_runs) if no_cutin_runs else 0.0

boundary = [r for r in cutin_runs
            if not r['collision'] and 1.5 <= r['min_ttc'] <= 2.5]

print('cut-in cases: %d' % len(cutin_runs))
print('passed: %d, pass rate = %.2f%%' % (len(passed), pass_rate * 100))
print('collisions: %d' % sum(1 for r in cutin_runs if r['collision']))
print('no-cut-in cases: %d, false triggers: %d, FTR = %.2f%%'
      % (len(no_cutin_runs), len(false_trig), ftr * 100))
print('boundary cases (1.5s<=TTC<=2.5s): %d' % len(boundary))
for r in boundary:
    print('  case %d: TTC=%.2fs, ego_v0=%.1fm/s, tgt_brake_v=%s'
          % (r['case'], r['min_ttc'], r['ego_v0'], r['tgt_brake_v']))

assert 0.30 <= pass_rate <= 0.90, \
    'pass rate %.2f%% out of [30%%, 90%%]!' % (pass_rate * 100)
assert ftr < 0.05, 'false trigger rate %.2f%% >= 5%%!' % (ftr * 100)
assert len(boundary) >= 1, 'no boundary case identified!'
print('')
print('ALL EVALUATIONS PASSED')


## 六、验证内容 | Verification Checklist
- [ ] 50 个变异场景全部运行，无 harness 错误
- [ ] 通过率在 30%–90% 之间（本轮实测值见上方输出）
- [ ] 误触发率 < 5%（无切入场景中不急刹车）
- [ ] 至少识别出 1 个边界案例（TTC 在 1.5–2.5s 之间）
- [ ] 碰撞案例的共同特征可解释（小切入间隙 + 目标车急减速）
- [ ] 能说出本实验的仿真引擎诚实声明（运动学模型，非 esmini 驱动）


## 七、结果分析 | Result Analysis

### 5.1 关键指标解读 | Key Metrics

| 指标 | 实测值 | 判据 | 结论 |
|---|---|---|---|
| 通过率 | 31.58% (12/38) | 30%-90% | ✅ |
| 误触发率 | 0.00% | < 5% | ✅ |
| 边界案例 | 4个 (TTC 1.74-2.32s) | ≥1个 | ✅ |
| 碰撞案例 | 24例 | — | 用于分析 |

**物理意义**：31.58%通过率说明变异强度适中——既有足够失败案例暴露问题，又有足够通过案例验证功能。0%误触发证明车道感知AEB逻辑有效。

### 5.2 结果合理性 | Rationality

通过率在30-90%区间内，符合模糊测试设计预期。4个边界案例（TTC 1.74-2.32s）正好落在"临界安全"区间，是最有价值的测试用例。

### 5.3 误差来源 | Error Sources

1. **变异分布**：正态扰动参数凭经验设定，可能未覆盖真实长尾
2. **判据阈值**：TTC>1.5s、FTR<5%为工程经验值，非法规强制
3. **模型简化**：纯Python运动学，未考虑轮胎非线性

### 5.4 工程意义 | Engineering Significance

模糊测试是发现"未知未知"风险的利器。24个碰撞案例中，可能藏着工程师没想到的场景组合。边界案例可直接转化为回归测试用例，纳入CI流水线。这是SOTIF（预期功能安全）的实践方法。

### 5.5 设计改进思考 | Design Improvement

- **从随机到引导**：纯随机变异效率低；**基于覆盖率的引导式模糊**（如 AFL 的思路）优先变异"带来新行为"的种子，发现失效更快。
- **从参数到轨迹**：本实验变异的是标量参数；真实 cut-in 的"目标车轨迹"本身可变异——**轨迹级变异**（如 B 样条扰动）更接近真实长尾。
- **从离线到 CI**：把模糊测试接入**持续集成**——每晚自动跑 500 个变异场景，回归守护 AEB 逻辑，这是 SOTIF"持续验证"理念的落地。



## 八、结果可视化 | Visualization
下图均为**真实 fuzz 运行数据**绘制：
- 左：最小 TTC 分布（绿=通过，红=失败/碰撞）
- 右：切入间隙 vs 最小 TTC（可看出间隙越小越危险）


In [ ]:
import matplotlib.pyplot as plt

ttc_pass = [r['min_ttc'] for r in passed]
ttc_pass = [t for t in ttc_pass if t < float('inf')]  # 未接近的案例min_ttc=inf（安全通过），直方图无法处理inf，与下方ttc_fail同口径剔除
ttc_fail = [r['min_ttc'] for r in cutin_runs if r not in passed]
ttc_fail = [t for t in ttc_fail if t < float('inf')]

plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.hist(ttc_pass, bins=12, alpha=0.7, label='Pass', color='green')
plt.hist(ttc_fail, bins=12, alpha=0.7, label='Fail/Collision', color='red')
plt.axvline(1.5, color='black', linestyle='--', label='Threshold 1.5s')
plt.xlabel('Min TTC (s)')
plt.ylabel('Count')
plt.title('Min TTC Distribution (50 fuzz cases)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for r in cutin_runs:
    if r['cutin_t'] is not None:
        dv = r['ego_v0'] - r['tgt_v0']
        g_est = (r['tgt_x0'] - r['ego_x0']) - dv * r['cutin_t']
        color = 'green' if r in passed else 'red'
        marker = 'x' if r['collision'] else 'o'
        ttc_plot = r['min_ttc'] if r['min_ttc'] < 20 else 20
        plt.scatter(g_est, ttc_plot, c=color, marker=marker, alpha=0.7)
plt.axhline(1.5, color='black', linestyle='--', label='Threshold 1.5s')
plt.xlabel('Cut-in gap (m)')
plt.ylabel('Min TTC (s, capped at 20)')
plt.title('Cut-in Gap vs Min TTC')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


**图意说明**：左为 50 个变异场景的最小 TTC 散点（红线=1.5s 判据线，线上通过、线下失效）；右为 TTC 分布直方图。双峰形态是典型结果：右峰是"轻松通过"的普通场景，左峰是"切得近"的危险场景——**左峰与判据线的交界区就是边界案例**（本实验 4 个），它们是回归测试最宝贵的用例。



## 九、挑战任务 | Challenges

1. **变异算子设计**：增加一个变异维度——目标车切入时的横向速度（本实验固定为正弦 3 秒完成）。
   - 横向速度越快，留给 AEB 的反应时间越短
   - 你的新算子会让通过率上升还是下降？

2. **控制器加固**：当前 AEB 在约 70% 的对抗场景下会碰撞。
   - 尝试实现"考虑目标车加速度"的 TTC（提示：解一元二次方程求碰撞时间）
   - 加固后通过率提升了多少？误触发率有变化吗？

3. **Oracle 深化**：本实验的通过判据是"无碰撞且 TTC>1.5s"。
   - 舒适性是否应该纳入判据？（如：最大减速度 < 6 m/s²）
   - 设计一个包含安全性+舒适性的综合评分函数

4. **种子敏感性**：把 `np.random.default_rng(42)` 改为 43、44，各跑一次。
   - 通过率稳定吗？这说明了什么？（提示：50 个样本是否足够）


---
## 阶段五：结果文件归档与平台官方评测 | Auto-Grading & Results Sync


In [ ]:
# 阶段五：结果文件归档与平台官方评测
import json, os
from common.evaluator import grade_metrics

record = {
    'experiment': 'H2_scenario_fuzzing',
    'metrics': {
        'pass_rate': round(float(pass_rate), 4),
        'false_trigger_rate': round(float(ftr), 4)
    }
}
saved_ok = False
for target_dir in ['/home/jovyan/labs/data/results', 'labs/data/results', 'results']:
    try:
        os.makedirs(target_dir, exist_ok=True)
        out_file = os.path.join(target_dir, 'H2_scenario_fuzzing.json')
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f'✓ 结果文件已成功归档至: {out_file}')
        saved_ok = True
        break
    except OSError:
        continue
eval_report = grade_metrics('H2_scenario_fuzzing', record['metrics'])
print(f'🏆 官方评分: {eval_report["score"]} / 100 分 ({'✅ 通过' if eval_report["passed"] else '❌ 未通过'})')


---

## 下一个实验 | Next →

🎉 恭喜完成全部29个实验！

> **所需内核**：`python3` | **Required kernel**: `python3`
